# 2 · Train and evaluate (15-minute horizon)
**Split by time:** training → 7-day validation → 7-day test (the newest week). Rows whose 15-minute label window crosses a cutoff are dropped. Thresholds and model choice use validation only; the test week is scored once.

**Models, per target (bike = runs out of bikes, dock = runs out of docks):**
| Model | What it is |
|---|---|
| `count_only` | Baseline: logistic regression on the number of bikes/docks left, nothing else |
| `history` | Baseline: this station's historical onset rate at this hour (weekday/weekend), from training data |
| `logreg_geo` | The professor's model: logistic regression + geographic-cell intercepts, penalty tuned by time-gap cross-validation |
| `gbm` | Gradient-boosted trees on the same predictors plus latitude/longitude |

"Same state as now" (persistence) isn't a usable baseline here: targets only exist for stations that aren't out yet, so persistence would always predict 0.

**Metrics:** log loss and Brier score (probability quality), PR-AUC (ranking for a rare event), precision/recall/F1 at the validation-chosen warning threshold, and calibration curves.

**Needs:** notebook 1. **Writes (production):** `team1_split_15`, `team1_model_eval_15`, `team1_calibration_15`, `team1_station_hour_rates_15`, `team1_model_info_15`, and models `team1_{bike,dock}_stockout_15@champion`.

In [ ]:
%pip install scikit-lego==0.9.9
dbutils.library.restartPython()

In [ ]:
# Settings, shared code, and run mode
from datetime import timedelta

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pyspark.sql import functions as F
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklego.model_selection import TimeGapSplit

import mvp_config as cfg
import mvp_lib as lib

spark.conf.set("spark.sql.session.timeZone", "UTC")
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
PUBLISH = dbutils.widgets.get("run_mode") == "production"

CURRENT_USER = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT = f"/Users/{CURRENT_USER}/team1-citibike-15min" + ("" if PUBLISH else "-dev")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT)
print(f"run_mode={dbutils.widgets.get('run_mode')} · experiment {EXPERIMENT}")

## 1. Chronological split

In [ ]:
keep = ["station_id", "fetched_at", "hour", "weekend", "geographic_cell", "lat", "lon",
        "bike_stockout", "dock_stockout",
        *sorted(set(lib.numeric_features("bike") + lib.numeric_features("dock")))]
data = (spark.table(cfg.PREDICTORS).select(*keep)
        .join(spark.table(cfg.TARGETS).select("station_id", "fetched_at", "bike_target",
                                               "dock_target", "label_end_microseconds"),
              ["station_id", "fetched_at"])
        .filter(F.col("bike_target").isNotNull() | F.col("dock_target").isNotNull()))

cutoffs = lib.split_cutoffs(spark.table(cfg.TARGETS))
data = lib.with_split(data, cutoffs)
print({k: str(v) for k, v in cutoffs.items()})

split_counts = (data.groupBy("split")
                .agg(F.count("bike_target").alias("bike_rows"), F.avg("bike_target").alias("bike_rate"),
                     F.count("dock_target").alias("dock_rows"), F.avg("dock_target").alias("dock_rate"))
                .toPandas().set_index("split").loc[["train", "validation", "test"]])
display(split_counts.reset_index())

## 2. History baseline: station × hour × weekend onset rate (all training rows)

In [ ]:
train_spark = data.filter("split = 'train'")
rates, overall = {}, {}
for t in ("bike", "dock"):
    rates[t], overall[t] = lib.station_hour_rates(train_spark, t)
    print(f"{t}: overall training onset rate {overall[t]:.4f}")

history_table = rates["bike"].join(rates["dock"], ["station_id", "hour", "weekend"], "full")
if PUBLISH:
    history_table.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(cfg.STATION_HISTORY)

## 3. Samples into pandas
Free Edition can't hold every row in memory, so training uses a random 5% sample (logistic regression has a few hundred coefficients, so this changes little) and metrics use 25% of validation and test.

In [ ]:
def to_pandas(frame, fraction):
    pdf = (frame.join(rates["bike"], ["station_id", "hour", "weekend"], "left")
                .join(rates["dock"], ["station_id", "hour", "weekend"], "left")
                .sample(fraction=fraction, seed=cfg.SEED).toPandas())
    numeric = [c for c in pdf.columns if c.startswith(("num_", "bike_", "dock_", "log_", "hour_", "history_"))
               and c not in ("bike_stockout", "dock_stockout")] + ["weekend", "lat", "lon"]
    pdf[numeric] = pdf[numeric].astype("float64")
    for t in ("bike", "dock"):
        pdf[f"history_rate_{t}"] = pdf[f"history_rate_{t}"].fillna(overall[t])
    return pdf.sort_values("fetched_at").reset_index(drop=True)

train_pd = to_pandas(data.filter("split = 'train'"), cfg.TRAIN_SAMPLE)
val_pd = to_pandas(data.filter("split = 'validation'"), cfg.EVAL_SAMPLE)
test_pd = to_pandas(data.filter("split = 'test'"), cfg.EVAL_SAMPLE)
print(f"pandas rows · train {len(train_pd):,} · validation {len(val_pd):,} · test {len(test_pd):,}")

## 4. Fit the models

In [ ]:
def make_models(target, train):
    """Fit count_only, logreg_geo (tuned), and gbm for one target."""
    num = lib.numeric_features(target)
    y = train[f"{target}_target"].astype(int)

    count_only = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                               LogisticRegression(max_iter=500))
    count_only.fit(train[[f"num_{target}s_available"]], y)

    geo = make_pipeline(
        ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["geographic_cell"]),
        ]),
        LogisticRegression(max_iter=300),
    )
    cv = TimeGapSplit(date_serie=train["fetched_at"], train_duration=timedelta(days=cfg.CV_TRAIN_DAYS),
                      valid_duration=timedelta(days=cfg.CV_VALID_DAYS),
                      gap_duration=timedelta(minutes=cfg.HORIZON_MIN + cfg.MAX_OBSERVATION_GAP_MIN),
                      window="expanding")
    search = GridSearchCV(geo, {"logisticregression__C": cfg.C_GRID}, cv=cv, scoring="neg_log_loss")
    search.fit(train[num + ["geographic_cell"]], y)

    gbm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early_stopping=True,
                                         random_state=cfg.SEED)
    gbm.fit(train[lib.gbm_features(target)], y)
    return {"count_only": count_only, "logreg_geo": search.best_estimator_, "gbm": gbm}, search


fitted, searches = {}, {}
for t in ("bike", "dock"):
    tr = train_pd[train_pd[f"{t}_target"].notna()].reset_index(drop=True)
    fitted[t], searches[t] = make_models(t, tr)
    print(f"{t}: {len(tr):,} training rows · best C = {searches[t].best_params_['logisticregression__C']} "
          f"· CV log loss {-searches[t].best_score_:.4f}")

## 5. Validation (choose) → test (report)

In [ ]:
def predictions(target, frame):
    """{model: probabilities} including the history baseline."""
    out = {"history": frame[f"history_rate_{target}"].to_numpy()}
    for name, model in fitted[target].items():
        out[name] = lib.predict_proba(model, frame[lib.model_columns(model)])
    return out

eval_rows, calib, chosen = [], [], {}
for t in ("bike", "dock"):
    va = val_pd[val_pd[f"{t}_target"].notna()]
    te = test_pd[test_pd[f"{t}_target"].notna()]
    pv, pt = predictions(t, va), predictions(t, te)
    yv, yt = va[f"{t}_target"].to_numpy(), te[f"{t}_target"].to_numpy()
    for model in pv:
        thr = lib.best_threshold(yv, pv[model])            # chosen on validation
        for split, y, p in (("validation", yv, pv[model]), ("test", yt, pt[model])):
            eval_rows.append({"target": t, "model": model, "split": split, **lib.score(y, p, thr)})
            calib.append(lib.calibration_rows(y, p, t, model, split))
    v = pd.DataFrame([r for r in eval_rows if r["target"] == t and r["split"] == "validation"
                      and r["model"] in fitted[t]])
    chosen[t] = v.loc[v["log_loss"].idxmin(), "model"]      # lowest validation log loss
    print(f"{t}: chosen model = {chosen[t]}")

results = pd.DataFrame(eval_rows)
calibration = pd.concat(calib, ignore_index=True)
display(results.round(4))

In [ ]:
# The MVP check: does the chosen model beat both baselines on the untouched test week?
for t in ("bike", "dock"):
    r = results[(results.target == t) & (results.split == "test")].set_index("model")
    m = chosen[t]
    print(f"{t}: {m}  log loss {r.loc[m, 'log_loss']:.4f} (history {r.loc['history', 'log_loss']:.4f}, "
          f"count-only {r.loc['count_only', 'log_loss']:.4f})  ·  PR-AUC {r.loc[m, 'pr_auc']:.3f} "
          f"(history {r.loc['history', 'pr_auc']:.3f}, count-only {r.loc['count_only', 'pr_auc']:.3f}, "
          f"base rate {r.loc[m, 'base_rate']:.3f})")

# Calibration on the test week: points on the diagonal = honest probabilities.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, t, label in zip(axes, ("bike", "dock"), ("Runs out of bikes", "Runs out of docks")):
    for model, g in calibration[(calibration.target == t) & (calibration.split == "test")].groupby("model"):
        ax.plot(g.mean_predicted, g.observed_rate, marker="o", label=model)
    lim = calibration.loc[calibration.target == t, ["mean_predicted", "observed_rate"]].max().max() * 1.05
    ax.plot([0, lim], [0, lim], "k--", lw=1)
    ax.set(title=f"{label} within {cfg.HORIZON_MIN} min — test week", xlabel="Mean predicted probability",
           ylabel="Observed rate", xlim=(0, lim), ylim=(0, lim))
    ax.legend()
fig.tight_layout()
display(fig)

## 6. Log to MLflow, register the chosen models, save results

In [ ]:
info_rows = []
stamp = pd.Timestamp.now(tz="UTC").tz_localize(None)
for t in ("bike", "dock"):
    m = chosen[t]
    model = fitted[t][m]
    cols = lib.model_columns(model)
    r = results[results.target == t].set_index(["split", "model"])
    with mlflow.start_run(run_name=f"team1_{t}_15min_{m}"):
        mlflow.log_params({"target": t, "horizon_min": cfg.HORIZON_MIN, "chosen_model": m,
                           "best_C": searches[t].best_params_["logisticregression__C"],
                           "train_sample": cfg.TRAIN_SAMPLE, "eval_sample": cfg.EVAL_SAMPLE,
                           "val_start": str(cutoffs["val_start"]), "test_start": str(cutoffs["test_start"])})
        for (split, model_name), row in r.iterrows():
            for k in ("log_loss", "brier", "pr_auc", "precision", "recall", "f1"):
                if pd.notna(row[k]):
                    mlflow.log_metric(f"{split}_{model_name}_{k}", float(row[k]))
        mlflow.log_metric("cv_log_loss", -searches[t].best_score_)
        mlflow.log_figure(fig, "calibration_test.png")
        example = val_pd[cols].head(5)
        info = mlflow.sklearn.log_model(
            model, name="model", input_example=example,
            signature=infer_signature(example, model.predict_proba(example)),
            pyfunc_predict_fn="predict_proba",
            registered_model_name=cfg.MODEL_NAMES[t] if PUBLISH else None)
    version = None
    if PUBLISH:
        version = int(info.registered_model_version)
        MlflowClient().set_registered_model_alias(cfg.MODEL_NAMES[t], cfg.MODEL_ALIAS, version)
        print(f"registered {cfg.MODEL_NAMES[t]} v{version} @{cfg.MODEL_ALIAS}")
    info_rows.append({
        "target": t, "model_name": cfg.MODEL_NAMES[t], "version": version, "model_type": m,
        "threshold": float(r.loc[("test", m), "threshold"]), "input_columns": ",".join(cols),
        "val_log_loss": float(r.loc[("validation", m), "log_loss"]),
        "test_log_loss": float(r.loc[("test", m), "log_loss"]),
        "test_pr_auc": float(r.loc[("test", m), "pr_auc"]),
        "test_history_log_loss": float(r.loc[("test", "history"), "log_loss"]),
        "test_history_pr_auc": float(r.loc[("test", "history"), "pr_auc"]),
        "test_count_only_pr_auc": float(r.loc[("test", "count_only"), "pr_auc"]),
        "test_base_rate": float(r.loc[("test", m), "base_rate"]),
        "val_start": cutoffs["val_start"], "test_start": cutoffs["test_start"],
        "data_end": cutoffs["data_end"], "trained_at": stamp,
    })

model_info = pd.DataFrame(info_rows)
display(model_info)

if PUBLISH:
    def save(pdf, table):
        spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    save(pd.DataFrame([{**{k: v for k, v in cutoffs.items()}, "horizon_min": cfg.HORIZON_MIN,
                        "created_at": stamp}]), cfg.SPLIT)
    save(results.assign(evaluated_at=stamp), cfg.EVAL)
    save(calibration, cfg.CALIBRATION)
    save(model_info.astype({"version": "Int64"}), cfg.MODEL_INFO)
    print("saved", cfg.SPLIT, cfg.EVAL, cfg.CALIBRATION, cfg.MODEL_INFO)
else:
    print("dev: models not registered, tables not written")